In [ ]:
# Local SapBERT Configuration
SAPBERT_MODEL_PATH = (
    "/workspaces/snomed_methods/embedding_models/SapBERT-from-PubMedBERT-fulltext"
)

In [ ]:
# Load MedCAT model pack
from medcat.cat import CAT

medcat_path = (
    "/workspaces/snomed_methods/model_packs/medcat_model_pack_422d1d38fc58f158.zip"
)

print(f"Loading MedCAT model pack: {medcat_path}")
cat = CAT.load_model_pack(medcat_path)
print("✓ Loaded MedCAT model")
print(f"  CDB has {len(cat.cdb.cui2preferred_name)} concepts")

In [ ]:
# Load the embedder module
import sys

sys.path.insert(0, "/workspaces/snomed_methods")

from llm_concept_embedder import (
    ClinicalConceptEmbedder,
    ConceptVectorSearch,
    load_concepts_from_medcat,
)

In [ ]:
# Initialize SapBERT embedder with Transformers backend
print("Initializing ClinicalConceptEmbedder with local SapBERT model...")
embedder = ClinicalConceptEmbedder(
    model_name_or_path=SAPBERT_MODEL_PATH, backend="transformers", device="cpu"
)
print("✓ Local SapBERT embedder initialized")
print(f"  Model path: {embedder.model_name_or_path}")

In [ ]:
# Configure output path
import os

output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)
embeddings_path = os.path.join(output_dir, "concept_embeddings.pkl")

In [ ]:
# Check if embeddings already exist, otherwise generate them
if os.path.exists(embeddings_path):
    print(f"Loading existing embeddings from: {embeddings_path}")
    cui_to_embedding = ConceptVectorSearch.load_embeddings_from_file(embeddings_path)
    print(f"✓ Loaded {len(cui_to_embedding)} pre-computed entities")
else:
    print("Loading SNOMED concepts from MedCAT CDB...")
    concept_df = load_concepts_from_medcat(cat)

    print(f"✓ Loaded {len(concept_df)} concepts")
    print("\nSample (first 3 concepts):")
    for i, row in concept_df.head(3).iterrows():
        print(f"  CUI: {row['cui']}, Name: {row['preferred_name']}")

In [ ]:
# Prepare concept texts (only if we need to generate new embeddings)
if not os.path.exists(embeddings_path):
    print("\nPreparing concept text prompts...")
    concept_texts = embedder.prepare_concept_text(concept_df)

    print(f"✓ Prepared {len(concept_texts)} text prompts")

In [ ]:
# Generate embeddings (only if needed)
if not os.path.exists(embeddings_path):
    print("\nGenerating embeddings with local SapBERT model...")

    embeddings = embedder.generate_embeddings(concept_texts, batch_size=32)

    print("✓ Generated embeddings")
    print(f"  Shape: {embeddings.shape}")

    # Create embedding dictionary with CUI as key
    cui_to_embedding = {}
    for i, cui in enumerate(concept_df["cui"].tolist()):
        cui_to_embedding[cui] = embeddings[i]

    # Save embeddings for reuse
    print(f"\nSaving embeddings to: {embeddings_path}")
    embedder.export_embeddings(cui_to_embedding, embeddings_path)
    print("✓ Embeddings saved for future use")

In [ ]:
# Build FAISS index
print("\nBuilding FAISS index...")

# Load or create names mapping (use CDB for this)
cui_to_name = {}
for cui in cui_to_embedding.keys():
    if hasattr(cat, "cdb") and hasattr(cat.cdb, "cui2preferred_name"):
        name = cat.cdb.cui2preferred_name.get(cui, f"CUI: {cui}")
        cui_to_name[cui] = name

search_engine = ConceptVectorSearch(
    {"embeddings": cui_to_embedding, "names": cui_to_name}, embedder=embedder
)
search_engine.build_index(index_type="FlatIP")

print(f"✓ Built FAISS index with {len(search_engine.cui_list)} concepts")

In [ ]:
# Query parameters
TEST_CONCEPT = "meningioma"
TOP_K_RESULTS = 10

# Query for similar concepts
print(f"\nQuerying for concepts similar to '{TEST_CONCEPT}'...")
query_results = search_engine.search(TEST_CONCEPT, top_k=TOP_K_RESULTS)

print(f"\n✓ Found {len(query_results)} similar concepts\n")
print("Top results:")
for i, (cui, name, score) in enumerate(query_results[:10], 1):
    print(f"{i:2d}. CUI: {cui:15s} | Score: {score:.4f}")
    print(f"     Name: {name}")